### 잔기 거리 정렬 엑셀 파일 산출( > 4 Å)

```mermaid

```

In [53]:
import os
os.chdir("/home/kjk/work") 

In [54]:
import csv
import numpy as np
from Bio.PDB import PDBParser

CUTOFF = 4.0
ANTIGEN = ["A"]
ANTIBODY = ["B", "C"]

model = PDBParser(QUIET=True).get_structure("c", "data/raw/5X8M.pdb")[0]


###### **heavy_atomes → PDB에서 물,리간드,수소 제외한 순수 아미노산의 원자 정보 수집**

In [ ]:
def heavy_atoms(model, chain_ids):        # model에 model 들어감, chain_ids에 ANTIGEN, ANTIBODY 들어감
    atoms = []
    for chain_id in chain_ids:
        for res in model[chain_id]:
            if res.id[0] != " ":
                continue
            for atom in res:
                if atom.element != "H":
                    atoms.append(atom)
    return atoms

###### **원자 정보 → 잔기 정보(chain.id, 잔기 번호, 잔기 이름) 튜플 수집**

In [56]:
def residue_key(atom):
    res = atom.get_parent()
    return (res.get_parent().id, res.id[1], res.get_resname())

###### **모든 원자간 거리 산출**

In [57]:
# 1) 원자 목록, 좌표 배열
ag_atoms = heavy_atoms(model, ANTIGEN)
ab_atoms = heavy_atoms(model, ANTIBODY)
X = np.array([a.coord for a in ag_atoms])
Y = np.array([b.coord for b in ab_atoms])

# 2) 모든 원자 쌍 거리: 배열
D = np.linalg.norm(X[:, None, :] - Y[None, :, :], axis=-1)

###### **기준 거리 이내인 쌍 배열 위치 산출**

In [58]:
ag_idx, ab_idx = np.where(D < CUTOFF)

###### **원자 배열 → 잔기 집합/엑셀 정보 산출**

In [59]:
epitope, paratope, rows = set(), set(), []
for i, j in zip(ag_idx, ab_idx):    # ag_idx, ab_idx는 배열이라서 리스트로 풀어야 함
    a, b = ag_atoms[i], ab_atoms[j]
    epitope.add(residue_key(a))
    paratope.add(residue_key(b))
    rows.append([*residue_key(a), a.get_id(), *residue_key(b), b.get_id(), round(float(D[i, j]), 2)])

###### **엑셀 생성 → 기입 → 저장**

In [60]:
rows.sort(key=lambda r: r[-1])     # 기존 def는 재사용을 위해 만들어두지만 그렇지 않고 일회용으로 쓰는 함수는 key=lamda r: return할 값 으로 사용
with open("results/5X8M_contacts.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["ag_chain", "ag_resnum", "ag_resname", "ag_atom",
                     "ab_chain", "ab_resnum", "ab_resname", "ab_atom", "distance"])
    writer.writerows(rows)
print("\nsaved results/5X8M_contacts.csv")


saved results/5X8M_contacts.csv
